In [1]:
import numpy as np
from IPython.display import Image, display
from sedona.spark import SedonaContext
import itertools
import os
import pandas as pd
import geopandas as gpd
from sedona.spark.stats.clustering.dbscan import dbscan
from sedona.spark.stats.autocorrelation.moran import Moran

In [2]:
%%capture
bucket_name = os.environ.get("SEDONA_SOURCE_BUCKET", "apache-sedona-book")

config = SedonaContext.builder()

sedona = SedonaContext.create(config.getOrCreate())

sedona.sparkContext.setLogLevel("ERROR")

sc = sedona.sparkContext
sedona.sparkContext.setCheckpointDir("checkpoint")

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/09/07 00:04:07 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
25/09/07 00:04:09 WARN UDTRegistration: Cannot register UDT for org.geotools.coverage.grid.GridCoverage2D, which is already registered.
25/09/07 00:04:09 WARN SimpleFunctionRegistry: The function rs_union_aggr replaced a previously registered function.
25/09/07 00:04:09 WARN UDTRegistration: Cannot register UDT for org.locationtech.jts.geom.Geometry, which is already registered.
25/09/07 00:04:09 WARN UDTRegistration: Cannot register UDT for org.apache.sedona.common.S2Geography.Geography, which is already registered.
25/09/07 00:04:09 WARN UDTRegistration: Cannot register UDT for org.locationtech.jts.index.SpatialIndex, which is already registered.
25/09/07 00:04:09 WARN SimpleFunctionRegistry: The function st_envelop

In [3]:
df = sedona.read.format("csv").\
    option("header", "true").\
    load(f"s3a://{bucket_name}/source_data/air_quality_moran").\
    where("Date = '12/05/2024'").\
    selectExpr(
        "monotonically_increasing_id() AS id",
        "ST_MakePoint(CAST(`Site Longitude` AS Double), CAST(`Site Latitude` AS double)) AS geom",
        "`Daily AQI Value` AS value"
    )

In [4]:
from sedona.spark.stats.weighting import add_distance_band_column

In [5]:
weights_df = add_distance_band_column(
    dataframe=df,
    threshold=1.0,
    include_self=True
)

In [6]:
# wait until the code will be merged

In [7]:
moran_i_result = Moran.get_global(weights_df)

In [13]:
moran_i_result.p_norm
moran_i_result.i
moran_i_result.z_norm

16.520562615514653

In [8]:
moran_i_result

MoranResult(i=0.6483597507522451, p_norm=0.0, z_norm=16.520562615514653)